# 08 · Final scoreboard (all notebooks)

Collects every `headline_*.csv`, `scoreboard_*.csv` and `fig_*.png` from the outputs of notebooks 01–07
(attach them as inputs) and writes one file, `final_numbers.md`, with every number the report quotes.
CPU only; no training.

In [ ]:
# Setup: make the `fusionlab` library importable. On Kaggle, always use a FRESH clone of the latest code (kept in /tmp,
# so it is not saved into this notebook's output); attached copies are used only when there is no Internet.
import glob, os, subprocess, sys

REPO_URL = 'https://github.com/fasihh/bigthing'      # the JF/ folder holds this project
BRANCH = 'jf-experiments'                           # used only until JF/ is merged into main


def fresh_clone(dest='/tmp/bigthing'):
    if not os.path.isdir(os.path.join(dest, '.git')):
        r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, dest], capture_output=True, text=True)
        if r.returncode:
            return None
    if not os.path.isdir(os.path.join(dest, 'JF')):  # JF not merged into main yet: use its branch
        subprocess.run(['git', '-C', dest, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', dest, 'checkout', '-q', 'FETCH_HEAD'], check=True)
    return os.path.join(dest, 'JF')


def find_jf():
    here = os.path.abspath('.')
    if not here.startswith('/kaggle'):                # running inside a local checkout of the repo
        for c in (os.path.dirname(here), here, os.path.join(here, 'JF')):
            if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
                return c
    jf = fresh_clone()
    if jf:
        return jf
    for c in glob.glob('/kaggle/input/**/JF', recursive=True):   # offline fallback (may be older code)
        if os.path.isdir(os.path.join(c, 'src', 'fusionlab')):
            print('WARNING: no Internet, using an attached copy of the code:', c)
            return c
    raise RuntimeError('Cannot find the fusionlab code: turn on Internet in the notebook settings.')


JF = find_jf()
sys.path.insert(0, os.path.join(JF, 'src'))
CONFIGS = os.path.join(JF, 'configs')
commit = subprocess.run(['git', '-C', JF, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('using fusionlab from', JF, '| commit', commit or 'n/a')

In [ ]:
import glob, shutil
import pandas as pd
from IPython.display import Image, Markdown, display

OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else './out_08'
ROOTS = ['/kaggle/input', os.path.join(JF, 'results')]
os.makedirs(os.path.join(OUT, 'figures'), exist_ok=True)


def find(pattern):
    hits = {}
    for r in ROOTS:
        for p in glob.glob(os.path.join(r, '**', pattern), recursive=True):
            hits.setdefault(os.path.basename(p), p)          # first copy of each file wins
    return dict(sorted(hits.items()))


heads, boards, figs = find('headline_*.csv'), find('scoreboard_*.csv'), find('fig_*.png')
print(len(heads), 'headline files,', len(boards), 'scoreboards,', len(figs), 'figures')

In [ ]:
TITLES = {'02': 'Huang (2026) reproduction with new seeds', '03': 'Missing modalities',
          '04': 'Corrupted and mismatched modalities', '05': 'Multimodal noisy TV (noise channel, distractor)',
          '06': 'Downstream decoders and calibration', '07': 'Novel methods'}
md = ['# Final numbers\n', 'Mean ± standard deviation over seeds unless stated. `*` marks the best value in a column.\n']
for name, p in heads.items():
    nb = name.split('_')[1].split('.')[0]
    md += [f'\n## {nb} · {TITLES.get(nb, "")}\n', pd.read_csv(p).to_markdown(index=False), '\n']
    for bn, bp in boards.items():
        if bn.startswith(f'scoreboard_{nb}') and not bn.endswith('_raw.csv'):
            md += [f'\n**{bn.replace(".csv", "")}**\n', pd.read_csv(bp).to_markdown(index=False), '\n']
    for fn, fp in figs.items():
        if fn.startswith(f'fig_{nb}'):
            shutil.copy(fp, os.path.join(OUT, 'figures', fn))
            md.append(f'\n![{fn}](figures/{fn})\n')
open(os.path.join(OUT, 'final_numbers.md'), 'w').write('\n'.join(md))
display(Markdown('\n'.join(md[:60])))